# Лабораторная работа № 3. Часть 2. Анализ циклического XOR

**Вариант № 10.** Дана учебная криптограмма из файла «ММБИС Варианты для лаб 3.xlsx». Открытый текст — длинное предложение на русском языке. Байты сообщения получены в CP1251 и обработаны циклическим XOR. Длина ключа меньше 10, а начальная фаза неизвестна.

При запуске из корня проекта криптограмма загружается из `data/lab_03_2_ciphertext.txt`.

## 1. Метод Касиски

Преобразуем hex в байты. Если одинаковые фрагменты открытого текста встретились на одинаковой фазе ключа, то соответствующие фрагменты шифротекста тоже совпадут. Расстояния между повторениями будут кратны периоду. Собираем расстояния между соседними вхождениями одинаковых трёхбайтовых фрагментов и считаем их делители от 2 до 9. Период 1 делит любое расстояние и потому сам по себе совпадения не объясняет.

In [1]:
from collections import Counter, defaultdict
from itertools import pairwise
from math import gcd

from pathlib import Path

ciphertext = bytes.fromhex(Path('data/lab_03_2_ciphertext.txt').read_text(encoding='ascii'))

positions = defaultdict(list)
for i in range(len(ciphertext) - 2):
    positions[ciphertext[i:i + 3]].append(i)

distances = [
    right - left
    for matches in positions.values()
    for left, right in pairwise(matches)
]
if not distances:
    raise ValueError('В учебном шифротексте не найдено повторяющихся троек')

votes = {length: sum(distance % length == 0 for distance in distances)
         for length in range(2, 10)}
period = max(votes, key=votes.get)
print('Объём шифротекста в байтах —', len(ciphertext))
print('Расстояние → частота —', dict(sorted(Counter(distances).items())))
print('Период → число кратных расстояний —', votes)
print(f'Выбранный период — {period}, НОД расстояний — {gcd(*distances)}')

Объём шифротекста в байтах — 360
Расстояние → частота — {45: 1, 65: 2, 75: 4, 85: 2, 100: 8, 115: 1, 130: 4, 140: 2, 145: 1, 155: 2, 170: 3, 195: 1, 220: 2, 245: 3, 260: 1, 275: 1}
Период → число кратных расстояний — {2: 20, 3: 6, 4: 13, 5: 38, 6: 0, 7: 5, 8: 0, 9: 1}
Выбранный период — 5, НОД расстояний — 5


## 2. Частотная гипотеза и открытый текст

Байты с одинаковым остатком индекса по модулю периода образуют столбец. Внутри столбца все байты обработаны одинаковым байтом ключа, поэтому частоты сохраняются. Проверяем гипотезу, что самый частый байт каждого столбца соответствует пробелу. Тогда байт ключа равен XOR этого байта с `0x20`. Частотная гипотеза эвристическая. Здесь её подтверждает связный русский текст, включая пробелы и пунктуацию.

In [2]:
effective_key = bytearray()
print('Столбец | частый байт | частота | байт ключа')
for column in range(period):
    (mode, count), = Counter(ciphertext[column::period]).most_common(1)
    key_byte = mode ^ ord(' ')
    effective_key.append(key_byte)
    print(f'{column:7} | 0x{mode:02x}       | {count:7} | 0x{key_byte:02x}')

effective_key = bytes(effective_key)

def apply_xor(data, key, shift=0):
    return bytes(value ^ key[(i + shift) % len(key)]
                 for i, value in enumerate(data))

plaintext_bytes = apply_xor(ciphertext, effective_key)
plaintext = plaintext_bytes.decode('cp1251')
print()
print('Ключевой цикл на первом байте —', effective_key.decode('cp1251'))
print('Восстановленный открытый текст')
print(plaintext)
assert period == gcd(*distances) == 5
assert plaintext.startswith('Он говорил, что,')
assert plaintext.endswith('Вот и конец.')
assert apply_xor(plaintext_bytes, effective_key) == ciphertext

Столбец | частый байт | частота | байт ключа
      0 | 0x17       |       8 | 0x37
      1 | 0xc7       |      17 | 0xe7
      2 | 0xd5       |      16 | 0xf5
      3 | 0xc4       |      12 | 0xe4
      4 | 0xd8       |      15 | 0xf8

Ключевой цикл на первом байте — 7зхдш
Восстановленный открытый текст
Он говорил, что, когда его вели на казнь, он не думал ни о чём, кроме того, чтобы не споткнуться на ступенях, и что ему казалось странным, что люди думают о великих вещах в такие минуты, тогда как он думал только о том, чтобы не упасть, и что даже в ту секунду, когда на него надевали повязку, он успел заметить, что на ней есть пятно, и подумать: Вот и конец.


## 3. Ключ и начальное смещение

Используем соглашение $C_i=P_i\oplus K_{(i+s)\bmod m}$. По криптограмме восстанавливается цикл, начиная с первого байта сообщения. Если циклически сдвинуть записанный ключ и согласованно изменить $s$, результат останется тем же. Перечисляем все эквивалентные пары и проверяем их повторным шифрованием. Смещения 0 и $m$ эквивалентны, поэтому в таблице даны различные классы смещений от 0 до $m-1$.

In [3]:
print('Смещение | ключ')
for shift in range(period):
    key = effective_key[-shift:] + effective_key[:-shift]
    assert key.decode('cp1251').isalnum()
    assert apply_xor(plaintext_bytes, key, shift) == ciphertext
    print(f'{shift:8} | {key.decode("cp1251")}')

Смещение | ключ
       0 | 7зхдш
       1 | ш7зхд
       2 | дш7зх
       3 | хдш7з
       4 | зхдш7


## Вывод

Метод Касиски дал период 5. Частоты в пяти столбцах восстановили ключевой цикл `7зхдш` и открытый текст. Пара `7зхдш` со смещением 0 воспроизводит исходную криптограмму. Для выбора одной исходной пары из пяти требуется дополнительное условие о начале записи ключа.